# Acoustic ID — Module 2B: Noise Simulator
### Acoustic ID-Based Vehicle Horn Detection for Urban Noise Pollution Control

---

**Scope of this module:**
- Reads clean `.wav` files from `acoustic_signals/` (Module 2 output)
- Generates 28 noisy test files per vehicle: 7 profiles × 4 dB levels
- Saves all files to `noisy_signals/` — Module 3 reads exclusively from here
- Zero database interaction — purely a filesystem operation
- Safe to re-run: existing files are skipped, missing files are regenerated

**Two categories of profiles:**

| Category | Profiles | What happens to OOK signal |
|---|---|---|
| Additive | traffic, rain, dj, procession, hawker, crowd | Noise added on top — ultrasonic ID preserved |
| Replacement | tamper | Clean signal discarded entirely — horn tone + noise only, zero ultrasonic content |

**Why replacement for tamper:** The tamper scenario simulates a vehicle whose Acoustic ID module
has been removed or jammed. The horn sounds normally but no OOK signal is emitted above 20 kHz.
Module 3 finds no preamble in the ultrasonic band and flags the event as `ANPR_FALLBACK` —
triggering the ₹5,000 tampering challan path instead of the normal ₹1,000 honking path.

**Libraries:** `numpy`, `scipy`, `soundfile` — no new installs required.

## Step 1: Import Libraries

In [ ]:
import numpy as np
import soundfile as sf
import os
import glob
from scipy import signal as sp_signal

## Step 2: Configuration

**Only the first two lines need to change between runs.**

`SAMPLE_RATE`, `SEED`, and `DB_LEVELS` must match Module 2 and Module 3 exactly.

`ADDITIVE_PROFILES` — noise is mixed on top of the clean OOK signal. Ultrasonic ID intact.

`REPLACEMENT_PROFILES` — clean signal is discarded. Output contains only horn tone + noise.
No preamble, no OOK encoding, no ultrasonic content. Module 3 will return `ANPR_FALLBACK`.

In [ ]:
# ── USER CONFIGURATION — change only these two lines ─────────────────────────
SIGNALS_DIR = "acoustic_signals"    # clean .wav input  (Module 2 output)
NOISY_DIR   = "noisy_signals"       # noisy .wav output (Module 3 input)
# ─────────────────────────────────────────────────────────────────────────────

SAMPLE_RATE          = 96000
SEED                 = 42
DB_LEVELS            = [60, 70, 80, 90]
# SNR map: dB labels → actual signal-to-noise ratios
SNR_MAP = {
    60: 20,    # SNR +20 dB (signal 10× louder than noise)
    70: 10,    # SNR +10 dB (signal 3× louder than noise)
    80: 0,     # SNR  0 dB  (signal = noise)
    90: -20,   # SNR -20 dB (noise 10× louder than signal)
}

# Additive: noise layered on top of clean OOK — ultrasonic ID preserved
ADDITIVE_PROFILES    = {"traffic", "rain", "dj", "procession", "hawker", "crowd"}

# Replacement: clean signal discarded — horn tone + noise only, no ultrasonic content
REPLACEMENT_PROFILES = {"tamper"}

# All profiles — populated after Step 4
SYNTHESISERS = {}

notebook_dir = os.getcwd()
signals_dir  = os.path.join(notebook_dir, SIGNALS_DIR)
noisy_dir    = os.path.join(notebook_dir, NOISY_DIR)

os.makedirs(noisy_dir, exist_ok=True)

print(f"Clean signals  : '{signals_dir}'")
print(f"Noisy output   : '{noisy_dir}'")
print(f"Noise levels   : {DB_LEVELS} dB")
print(f"Additive       : {sorted(ADDITIVE_PROFILES)}")
print(f"Replacement    : {sorted(REPLACEMENT_PROFILES)}")
print(f"Files/vehicle  : {len(DB_LEVELS) * (len(ADDITIVE_PROFILES) + len(REPLACEMENT_PROFILES))}")

## Step 3: Core Utilities

**`bandlimited_noise`** — Gaussian noise filtered to a frequency band using a 4th-order
Butterworth filter in SOS (second-order sections) form. SOS form is numerically stable
at low frequencies — BA form overflows at 60–200 Hz (DJ bass range).
Output is always normalised to unit peak.

**`measure_db` / `scale_to_db`** — RMS dB measurement and precise scaling.
Noise is scaled to target dB *before* mixing — the dB value refers to the noise floor
alone, not the combined signal.

**`mix_noise`** — scales noise to target dB, adds to base signal, clips to `[-1.0, 1.0]`.
Used by both additive profiles (base = clean OOK) and replacement profiles (base = horn tone).

In [ ]:
def bandlimited_noise(
    n_samples : int,
    low_hz    : float,
    high_hz   : float,
    sr        : int,
    rng       : np.random.Generator
) -> np.ndarray:
    """
    Gaussian noise bandlimited to [low_hz, high_hz] Hz.
    SOS-form Butterworth filter — numerically stable at all frequencies.
    Output normalised to unit peak amplitude.

    Args:
        n_samples (int)            : Number of samples
        low_hz    (float)          : Lower cutoff in Hz
        high_hz   (float)          : Upper cutoff in Hz
        sr        (int)            : Sample rate in Hz
        rng       (np.random.Generator): Seeded RNG

    Returns:
        np.ndarray: float32, peak-normalised to [-1.0, 1.0]
    """
    noise = rng.standard_normal(n_samples).astype(np.float64)
    nyq   = sr / 2.0
    l     = np.clip(low_hz  / nyq, 1e-4, 0.9999)
    h     = np.clip(high_hz / nyq, 1e-4, 0.9999)
    if l >= h:
        p = np.max(np.abs(noise))
        return (noise / p if p > 0 else noise).astype(np.float32)
    sos      = sp_signal.butter(4, [l, h], btype="band", output="sos")
    filtered = sp_signal.sosfilt(sos, noise)
    peak     = np.max(np.abs(filtered))
    return (filtered / peak if peak > 0 else filtered).astype(np.float32)


def measure_db(signal: np.ndarray) -> float:
    """
    RMS power of a signal in dB.

    Returns:
        float: RMS dB, or -inf if silent
    """
    rms = np.sqrt(np.mean(signal.astype(np.float64) ** 2))
    return 20 * np.log10(rms) if rms > 1e-12 else -np.inf


def scale_to_db(signal: np.ndarray, target_db: float) -> np.ndarray:
    """
    Scale a signal to a target RMS dB level.

    Args:
        signal    (np.ndarray): Input (should be peak-normalised)
        target_db (float)     : Target RMS level in dB

    Returns:
        np.ndarray: float32 scaled to target_db
    """
    current = measure_db(signal)
    if current == -np.inf:
        return signal.copy()
    gain = 10 ** ((target_db - current) / 20)
    return (signal.astype(np.float64) * gain).astype(np.float32)


def mix_noise(
    base      : np.ndarray,
    noise     : np.ndarray,
    target_db : float
) -> np.ndarray:
    """
    Mix noise into base signal at a specified SNR level.

    The target_db value is mapped to an SNR condition via SNR_MAP.
    Noise is scaled relative to the base signal's RMS — not to an
    absolute dB level — so the OOK signal is preserved at all but
    the most extreme noise conditions.

    Args:
        base      (np.ndarray): Base signal (clean OOK or horn tone)
        noise     (np.ndarray): Synthesised noise (peak-normalised)
        target_db (float)     : dB label (60/70/80/90) mapped to SNR

    Returns:
        np.ndarray: float32, clipped to [-1.0, 1.0]
    """
    snr_db    = SNR_MAP.get(int(target_db), 0)
    base_rms  = np.sqrt(np.mean(base.astype(np.float64) ** 2))
    noise_rms = np.sqrt(np.mean(noise.astype(np.float64) ** 2))

    if noise_rms < 1e-12 or base_rms < 1e-12:
        return np.clip(base, -1.0, 1.0).astype(np.float32)

    # Target noise RMS = base RMS / 10^(SNR/20)
    target_noise_rms = base_rms / (10 ** (snr_db / 20))
    gain             = target_noise_rms / noise_rms

    mixed = base.astype(np.float64) + noise.astype(np.float64) * gain
    return np.clip(mixed, -1.0, 1.0).astype(np.float32)

## Step 4: Noise Profile Synthesisers

All 7 synthesisers share the signature `synthesise_X(n_samples, sr, rng) -> np.float32`.
All outputs are peak-normalised before returning.

**Profiles 1–6 (additive)** — noise layered on top of the clean OOK signal:

| Profile | Frequency shape | Modulation | Scenario |
|---|---|---|---|
| `traffic` | 80–6000 Hz layered | Slow swell 0.2 Hz | Rush hour, major intersection |
| `rain` | Full spectrum, rolloff >10 kHz | Steady | Heavy monsoon |
| `dj` | 60–2000 Hz bass-heavy | None — sustained | DJ event outside hospital |
| `procession` | 80–8000 Hz broadband | Rhythmic 2 Hz (dhol beat) | Wedding procession |
| `hawker` | 300–3400 Hz voice band | Speech rhythm 5 Hz | Street vendor loudspeaker |
| `crowd` | 300–4000 Hz multi-layer | Irregular overlapping AM | Rally, market, protest |

**Profile 7 (replacement)** — completely replaces the clean signal:

| Profile | Content | Ultrasonic | Module 3 result |
|---|---|---|---|
| `tamper` | Horn tone (420/840/1260 Hz) + traffic noise | Zero | `ANPR_FALLBACK` |

In [ ]:
# ── Profiles 1–6: Additive ───────────────────────────────────────────────────

def synthesise_traffic(n_samples, sr, rng):
    """
    Urban traffic: engine rumble (80-500 Hz) + tyre noise (500-2000 Hz)
    + road surface (2-6 kHz). Slow amplitude swell simulates passing vehicles.
    Scenario: dense urban traffic, rush hour, major intersection.
    """
    engine = bandlimited_noise(n_samples, 80,   500,  sr, rng) * 0.6
    tyre   = bandlimited_noise(n_samples, 500,  2000, sr, rng) * 0.3
    road   = bandlimited_noise(n_samples, 2000, 6000, sr, rng) * 0.1
    mixed  = (engine + tyre + road).astype(np.float64)
    t      = np.arange(n_samples) / sr
    mixed *= (0.7 + 0.3 * np.sin(2 * np.pi * 0.2 * t))
    peak   = np.max(np.abs(mixed))
    return (mixed / peak if peak > 0 else mixed).astype(np.float32)


def synthesise_rain(n_samples, sr, rng):
    """
    Rain: full-spectrum white noise with gentle rolloff above 10 kHz.
    Most spectrally uniform profile — hardest to separate from signal.
    Scenario: heavy monsoon, extreme downpour.
    """
    white  = rng.standard_normal(n_samples).astype(np.float64)
    sos    = sp_signal.butter(2, 10000 / (sr / 2), btype="low", output="sos")
    result = sp_signal.sosfilt(sos, white)
    peak   = np.max(np.abs(result))
    return (result / peak if peak > 0 else result).astype(np.float32)


def synthesise_dj(n_samples, sr, rng):
    """
    DJ / loud music: heavy bass (60-200 Hz) dominant, mid (200-800 Hz)
    and harmonic content (800-2000 Hz) at lower levels. Sustained.
    Scenario: DJ event or PA speaker adjacent to silent zone.
    """
    bass   = bandlimited_noise(n_samples, 60,  200,  sr, rng) * 0.7
    mid    = bandlimited_noise(n_samples, 200, 800,  sr, rng) * 0.2
    harm   = bandlimited_noise(n_samples, 800, 2000, sr, rng) * 0.1
    mixed  = (bass + mid + harm).astype(np.float64)
    peak   = np.max(np.abs(mixed))
    return (mixed / peak if peak > 0 else mixed).astype(np.float32)


def synthesise_procession(n_samples, sr, rng):
    """
    Procession / dhol: broadband noise (80-8000 Hz) with rhythmic amplitude
    envelope at 2 Hz (dhol beat) plus off-beat accent.
    Scenario: wedding procession or religious band near hospital gate.
    """
    broad  = bandlimited_noise(n_samples, 80, 8000, sr, rng).astype(np.float64)
    t      = np.arange(n_samples) / sr
    env    = np.abs(np.sin(2 * np.pi * 2.0 * t)) ** 0.3
    env   += 0.4 * np.abs(np.sin(2 * np.pi * 2.0 * t + np.pi * 0.6)) ** 0.5
    mixed  = broad * np.clip(env, 0.1, 1.0)
    peak   = np.max(np.abs(mixed))
    return (mixed / peak if peak > 0 else mixed).astype(np.float32)


def synthesise_hawker(n_samples, sr, rng):
    """
    Street hawker: voice-band noise (300-3400 Hz) with speech-rhythm
    amplitude modulation at 5 Hz and 1.3 Hz cadence.
    Scenario: street vendor with loudspeaker near school or court.
    """
    voice  = bandlimited_noise(n_samples, 300, 3400, sr, rng).astype(np.float64)
    t      = np.arange(n_samples) / sr
    mod    = 0.5 + 0.5 * np.sin(2*np.pi*5.0*t) * np.sin(2*np.pi*1.3*t)
    mixed  = voice * np.clip(mod, 0.05, 1.0)
    peak   = np.max(np.abs(mixed))
    return (mixed / peak if peak > 0 else mixed).astype(np.float32)


def synthesise_crowd(n_samples, sr, rng):
    """
    Crowd noise: multiple overlapping voice-band layers (300-4000 Hz)
    with irregular combined amplitude from slow sinusoids.
    Scenario: protest, rally, or dense market crowd near silent zone.
    """
    v1     = bandlimited_noise(n_samples, 300,  2000, sr, rng) * 0.4
    v2     = bandlimited_noise(n_samples, 500,  3400, sr, rng) * 0.3
    chat   = bandlimited_noise(n_samples, 1000, 4000, sr, rng) * 0.2
    amb    = bandlimited_noise(n_samples, 200,  800,  sr, rng) * 0.1
    mixed  = (v1 + v2 + chat + amb).astype(np.float64)
    t      = np.arange(n_samples) / sr
    irr    = 0.5 + 0.3*np.sin(2*np.pi*0.7*t) + 0.2*np.sin(2*np.pi*1.9*t)
    mixed *= np.clip(irr, 0.1, 1.0)
    peak   = np.max(np.abs(mixed))
    return (mixed / peak if peak > 0 else mixed).astype(np.float32)


# ── Profile 7: Replacement ────────────────────────────────────────────────────

def generate_horn_tone(n_samples: int, sr: int) -> np.ndarray:
    """
    Synthetic vehicle horn: 420 Hz fundamental + 840 Hz + 1260 Hz harmonics.
    Occupies the middle 60% of the signal with 5ms linear ramp onset/offset
    to prevent click artefacts. Zero ultrasonic content.

    Args:
        n_samples (int): Number of samples
        sr        (int): Sample rate in Hz

    Returns:
        np.ndarray: float32 horn tone, peak-normalised
    """
    t     = np.arange(n_samples) / sr
    horn  = (0.6 * np.sin(2*np.pi*420*t)
           + 0.3 * np.sin(2*np.pi*840*t)
           + 0.1 * np.sin(2*np.pi*1260*t)).astype(np.float64)
    start      = int(n_samples * 0.20)
    end        = int(n_samples * 0.80)
    ramp_samp  = int(sr * 0.005)
    envelope   = np.zeros(n_samples)
    envelope[start : start + ramp_samp]         = np.linspace(0, 1, ramp_samp)
    envelope[start + ramp_samp : end - ramp_samp] = 1.0
    envelope[end - ramp_samp : end]             = np.linspace(1, 0, ramp_samp)
    horn  *= envelope
    peak   = np.max(np.abs(horn))
    return (horn / peak if peak > 0 else horn).astype(np.float32)


def synthesise_tamper(n_samples: int, sr: int, rng: np.random.Generator) -> np.ndarray:
    """
    Tamper test signal: audible horn tone + traffic-like background noise.
    Contains ZERO ultrasonic content above 20 kHz.

    Simulates a vehicle whose Acoustic ID module has been removed, jammed,
    or is otherwise non-functional. The horn is audible and detectable,
    but the roadside unit finds no preamble in the ultrasonic band.

    Module 3 pipeline result: detection_mode = "ANPR_FALLBACK"
    Module 4 action: tampering challan ₹5,000 + mandatory RTO visit

    Note: noise level applied externally by mix_noise() — this function
    returns the base tamper signal at unit peak, same as all other synthesisers.
    """
    horn    = generate_horn_tone(n_samples, sr)
    bg      = (bandlimited_noise(n_samples, 80,  500,  sr, rng) * 0.6 +
               bandlimited_noise(n_samples, 500, 2000, sr, rng) * 0.4).astype(np.float64)
    p       = np.max(np.abs(bg))
    bg      = (bg / p if p > 0 else bg).astype(np.float32)
    combined = horn.astype(np.float64) * 0.7 + bg.astype(np.float64) * 0.3
    peak     = np.max(np.abs(combined))
    return (combined / peak if peak > 0 else combined).astype(np.float32)


# Register all synthesisers
SYNTHESISERS = {
    "traffic"    : synthesise_traffic,
    "rain"       : synthesise_rain,
    "dj"         : synthesise_dj,
    "procession" : synthesise_procession,
    "hawker"     : synthesise_hawker,
    "crowd"      : synthesise_crowd,
    "tamper"     : synthesise_tamper,
}

total_profiles = len(SYNTHESISERS)
total_per_vehicle = total_profiles * len(DB_LEVELS)
print(f"Profiles registered  : {list(SYNTHESISERS.keys())}")
print(f"Files per vehicle    : {total_per_vehicle}")

## Step 5: Skip Logic

Pure `os.path.exists()` check. No database, no state.
Every run checks the filesystem — if the file is there, skip. If missing, generate.

In [ ]:
def needs_generation(
    plate    : str,
    profile  : str,
    db_level : int,
    noisy_dir: str
) -> bool:
    """
    Return True if the noisy WAV file does not exist on disk.

    Args:
        plate     (str): Vehicle plate number
        profile   (str): Noise profile name
        db_level  (int): Target dB level
        noisy_dir (str): Path to noisy_signals/ folder

    Returns:
        bool: True if file needs to be generated
    """
    fname = f"{plate}_{profile}_{db_level}db.wav"
    return not os.path.exists(os.path.join(noisy_dir, fname))

## Step 6: Single Vehicle Generator

`generate_noisy_files()` processes one vehicle — generates all 28 noisy files.

**Additive path** (profiles 1–6): `mix_noise(clean_signal, noise, db)` —
noise added on top of OOK signal. Ultrasonic ID survives.

**Replacement path** (tamper): `mix_noise(tamper_signal, noise, db)` —
the clean OOK signal is never used. Output is horn tone + background noise only.

**Fallbacks:**
- Source WAV missing → returns error for this vehicle, other vehicles unaffected
- Source WAV unreadable → returns error, no crash
- Sample rate mismatch → caught before any processing
- Individual file write failure → logged, remaining files continue

In [ ]:
def generate_noisy_files(
    plate         : str,
    clean_wav_path: str,
    noisy_dir     : str,
    rng           : np.random.Generator
) -> dict:
    """
    Generate all 28 noisy WAV files for a single vehicle.

    Args:
        plate          (str): Vehicle plate number
        clean_wav_path (str): Path to clean .wav from Module 2
        noisy_dir      (str): Output folder
        rng (np.random.Generator): Shared seeded RNG from batch runner

    Returns:
        dict: {plate, encoded: [...], skipped: [...], failed: [...]}\n    """
    # ── Fallback 1: source file missing ──────────────────────────────────────
    if not os.path.exists(clean_wav_path):
        return {
            "plate"  : plate, "encoded": [], "skipped": [],
            "failed" : [{"file": "source",
                         "error": f"Clean WAV not found: '{clean_wav_path}'"}]
        }

    # ── Fallback 2: unreadable file ───────────────────────────────────────────
    try:
        clean_signal, sr = sf.read(clean_wav_path, dtype="float32")
    except Exception as e:
        return {
            "plate"  : plate, "encoded": [], "skipped": [],
            "failed" : [{"file": "source", "error": f"Failed to read WAV: {e}"}]
        }

    # ── Fallback 3: sample rate mismatch ─────────────────────────────────────
    if sr != SAMPLE_RATE:
        return {
            "plate"  : plate, "encoded": [], "skipped": [],
            "failed" : [{"file": "source",
                         "error": f"Sample rate {sr} Hz — expected {SAMPLE_RATE} Hz"}]
        }

    n_samples                = len(clean_signal)
    encoded, skipped, failed = [], [], []

    for profile, fn in SYNTHESISERS.items():
        for db in DB_LEVELS:
            fname = f"{plate}_{profile}_{db}db.wav"
            fpath = os.path.join(noisy_dir, fname)

            if not needs_generation(plate, profile, db, noisy_dir):
                skipped.append(fname)
                continue

            # ── Fallback 4: synthesis or write failure ────────────────────────
            try:
                # Per-file seed ensures each file is independently reproducible
                file_rng = np.random.default_rng(
                    int(__import__('hashlib').md5(
                        f'{plate}{profile}{db}'.encode()
                    ).hexdigest()[:8], 16)
                )
                noise = fn(n_samples, SAMPLE_RATE, file_rng)

                if profile in REPLACEMENT_PROFILES:
                    # Tamper: save the tamper signal at unit amplitude.
                    # Do NOT scale to dB — scale_to_db applies a massive gain
                    # that amplifies floating-point noise into the ultrasonic band.
                    # Tamper detection is binary (OOK present / absent), not
                    # SNR-dependent, so dB variation is meaningless for this profile.
                    # The 4 files are acoustically identical but labelled by dB
                    # for table consistency. Module 3 returns ANPR_FALLBACK for all.
                    mixed = np.clip(noise, -1.0, 1.0).astype(np.float32)
                else:
                    # Additive: noise layered on top of clean OOK signal
                    mixed = mix_noise(clean_signal, noise, db)

                sf.write(fpath, mixed, SAMPLE_RATE, subtype="FLOAT")
                encoded.append(fname)

            except Exception as e:
                failed.append({"file": fname, "error": str(e)})

    return {"plate": plate, "encoded": encoded, "skipped": skipped, "failed": failed}

## Step 7: Batch Runner

Scans `acoustic_signals/` for all `*_acoustic_id.wav` files, extracts plate numbers,
and processes each vehicle. A single shared `rng` instance is used across all vehicles —
fixed seed guarantees identical output every run.

If `acoustic_signals/` is empty or missing, a clear message is printed — no crash.

In [ ]:
def generate_all(
    signals_dir : str,
    noisy_dir   : str
) -> dict:
    """
    Generate all noisy WAV files for every vehicle in acoustic_signals/.

    Args:
        signals_dir (str): Path to acoustic_signals/ (Module 2 output)
        noisy_dir   (str): Path to noisy_signals/ output folder

    Returns:
        dict: {vehicles_found, total_encoded, total_skipped, total_failed, per_vehicle}
    """
    empty_result = {
        "vehicles_found": 0, "total_encoded": 0,
        "total_skipped" : 0, "total_failed" : 0, "per_vehicle": []
    }

    # ── Fallback: source folder missing ──────────────────────────────────────
    if not os.path.exists(signals_dir):
        print(f"Source folder not found: '{signals_dir}'")
        print("Run Module 2 first to generate clean WAV files.")
        return empty_result

    clean_wavs = sorted(glob.glob(os.path.join(signals_dir, "*_acoustic_id.wav")))

    # ── Fallback: no clean WAV files found ───────────────────────────────────
    if not clean_wavs:
        print(f"No *_acoustic_id.wav files found in '{signals_dir}'")
        print("Run Module 2 first to generate clean WAV files.")
        return empty_result

    rng = np.random.default_rng(SEED)

    per_vehicle = []
    total_encoded = total_skipped = total_failed = 0

    for wav_path in clean_wavs:
        plate  = os.path.basename(wav_path).replace("_acoustic_id.wav", "")
        result = generate_noisy_files(plate, wav_path, noisy_dir, rng)
        per_vehicle.append(result)
        total_encoded += len(result["encoded"])
        total_skipped += len(result["skipped"])
        total_failed  += len(result["failed"])

    return {
        "vehicles_found" : len(clean_wavs),
        "total_encoded"  : total_encoded,
        "total_skipped"  : total_skipped,
        "total_failed"   : total_failed,
        "per_vehicle"    : per_vehicle
    }

## Step 8: Run

Processes all vehicles found in `acoustic_signals/`. Safe to re-run — existing files are skipped.

In [ ]:
results = generate_all(signals_dir, noisy_dir)

print(f"Vehicles found     : {results['vehicles_found']}")
print(f"Files encoded      : {results['total_encoded']}")
print(f"Files skipped      : {results['total_skipped']}")
print(f"Failures           : {results['total_failed']}")

if results["vehicles_found"] > 0:
    print()
    hdr = f"  {'Plate':<14}  {'Encoded':>8}  {'Skipped':>8}  {'Failed':>8}"
    print(hdr)
    print(f"  {'-'*14}  {'-'*8}  {'-'*8}  {'-'*8}")
    for v in results["per_vehicle"]:
        print(
            f"  {v['plate']:<14}"
            f"  {len(v['encoded']):>8}"
            f"  {len(v['skipped']):>8}"
            f"  {len(v['failed']):>8}"
        )

if results["total_failed"] > 0:
    print()
    print("Failed files:")
    for v in results["per_vehicle"]:
        for f in v["failed"]:
            print(f"  [{v['plate']}] {f['file']} — {f['error']}")

## Step 9: Verification Utility

Call manually before running Module 3. Checks completeness and readability of every file.
Reports per vehicle: how many of the expected 28 files exist, which are missing, and whether each file loads cleanly.

In [ ]:
def verify_noisy_dir(signals_dir: str, noisy_dir: str) -> None:
    """
    Verify all noisy WAV files exist and are readable.
    Call manually before running Module 3.

    Args:
        signals_dir (str): acoustic_signals/ — used to find plate list
        noisy_dir   (str): noisy_signals/ — files to verify
    """
    clean_wavs = sorted(glob.glob(os.path.join(signals_dir, "*_acoustic_id.wav")))
    if not clean_wavs:
        print("No clean WAV files found — run Module 2 first.")
        return

    expected_per_vehicle = len(SYNTHESISERS) * len(DB_LEVELS)
    total_expected       = len(clean_wavs) * expected_per_vehicle
    total_present        = 0
    total_missing        = 0
    total_unreadable     = 0

    print(f"Verifying {len(clean_wavs)} vehicles × "
          f"{expected_per_vehicle} files = {total_expected} expected")
    print(f"Profiles: additive={sorted(ADDITIVE_PROFILES)}, "
          f"replacement={sorted(REPLACEMENT_PROFILES)}")
    print()

    for wav_path in clean_wavs:
        plate      = os.path.basename(wav_path).replace("_acoustic_id.wav", "")
        missing    = []
        unreadable = []

        for profile in SYNTHESISERS:
            for db in DB_LEVELS:
                fname = f"{plate}_{profile}_{db}db.wav"
                fpath = os.path.join(noisy_dir, fname)

                if not os.path.exists(fpath):
                    missing.append(fname)
                    continue

                try:
                    w, sr = sf.read(fpath, dtype="float32")
                    assert sr == SAMPLE_RATE
                    assert w.dtype == np.float32
                    assert not np.any(np.isnan(w))
                    assert not np.any(np.isinf(w))
                    assert np.max(np.abs(w)) <= 1.0
                    total_present += 1
                except Exception as e:
                    unreadable.append(f"{fname}: {e}")

        status = "OK" if not missing and not unreadable else "INCOMPLETE"
        present_count = expected_per_vehicle - len(missing)
        print(f"  {plate:<14} {present_count:>3}/{expected_per_vehicle} files  [{status}]")

        if missing:
            shown = missing[:4]
            extra = f" ... +{len(missing)-4} more" if len(missing) > 4 else ""
            print(f"    Missing: {shown}{extra}")
        for u in unreadable:
            print(f"    Unreadable: {u}")

        total_missing    += len(missing)
        total_unreadable += len(unreadable)

    print()
    print(f"Total present    : {total_present} / {total_expected}")
    print(f"Total missing    : {total_missing}")
    print(f"Total unreadable : {total_unreadable}")
    print()
    if total_missing == 0 and total_unreadable == 0:
        print("All files verified. noisy_signals/ is ready for Module 3.")
    else:
        print("Incomplete. Re-run Step 8 to generate missing files.")